In [ ]:
# 第一次运行自动补齐缺失包；已安装的不会重复安装。
import importlib.util
import os
import subprocess
import sys

PACKAGE_MAP = {
    "numpy": "numpy>=1.24,<2.3", "pandas": "pandas>=2.0,<2.4",
    "matplotlib": "matplotlib>=3.7,<3.11",
    "gradio": "gradio==4.44.1", "fastapi": "fastapi>=0.115.2,<0.116",
    "starlette": "starlette>=0.40,<0.47", "pydantic": "pydantic>=2,<2.12",
    "huggingface_hub": "huggingface-hub>=0.34,<1",
}
# 默认直接运行本地 Qwen；需要无模型快速演示时再改为 False。
RUN_LOCAL_MODEL = True
RUN_LOCAL_MODEL = RUN_LOCAL_MODEL or os.getenv("WRITING_COACH_USE_LOCAL_MODEL", "0") == "1"
# 0.5B 适合课堂速度；产品演示若内存和时间允许，可改为 Qwen/Qwen2.5-1.5B-Instruct 提高结构化输出成功率。
MODEL_ID = os.getenv("WRITING_COACH_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
if RUN_LOCAL_MODEL:
    PACKAGE_MAP.update({"torch": "torch>=2.1,<2.8", "transformers": "transformers>=4.45,<5",
                        "accelerate": "accelerate>=0.29,<2"})
missing = [req for module, req in PACKAGE_MAP.items() if importlib.util.find_spec(module) is None]
# Gradio 4 与 Transformers 4 共用 huggingface-hub 0.x，避免依赖版本冲突。
try:
    from importlib.metadata import version
    from packaging.version import Version
    hub_version = Version(version("huggingface-hub"))
    if not (Version("0.34") <= hub_version < Version("1.0")):
        missing.append("huggingface-hub>=0.34,<1")
    if importlib.util.find_spec("gradio") and Version(version("gradio")) != Version("4.44.1"):
        missing.append("gradio==4.44.1")
    if not (Version("0.115.2") <= Version(version("fastapi")) < Version("0.116")):
        missing.append("fastapi>=0.115.2,<0.116")
    if not (Version("0.40") <= Version(version("starlette")) < Version("0.47")):
        missing.append("starlette>=0.40,<0.47")
    if not (Version("2") <= Version(version("pydantic")) < Version("2.12")):
        missing.append("pydantic>=2,<2.12")
    if RUN_LOCAL_MODEL and importlib.util.find_spec("transformers") and Version(version("transformers")) >= Version("5"):
        missing.append("transformers>=4.45,<5")
except Exception:
    pass
if missing:
    print("安装缺失依赖:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--index-url", "https://pypi.org/simple", *missing])
else:
    print("✅ 第 4 课依赖已就绪")

✅ 第 4 课依赖已就绪


In [11]:
from __future__ import annotations
import html
import json
import os
import re
import time
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Protocol

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print("项目目录:", PROJECT_ROOT.resolve())

项目目录: D:\CodeData\Program Coding\Project\Writing_Coach_Agent


In [12]:
def extract_json(text: str) -> dict[str, Any]:
    cleaned = re.sub(r"^```(?:jsonl)?\s*|\s*```$", "", text.strip(), flags=re.I)
    try:
        value = json.loads(cleaned)
        if isinstance(value, dict): return value
    except json.JSONDecodeError:
        pass
    decoder = json.JSONDecoder()
    for i, char in enumerate(cleaned):
        if char == "{":
            try:
                value, _ = decoder.raw_decode(cleaned[i:])
                if isinstance(value, dict): return value
            except json.JSONDecodeError:
                continue
    raise ValueError("模型未返回合法 JSON")


class JSONBackend(Protocol):
    name: str
    def generate_json(self, system: str, user: str, schema: dict[str, Any]) -> dict[str, Any]: ...

class LocalQwenBackend:
    def __init__(self, model_id="Qwen/Qwen2.5-0.5B-Instruct"):
        self.model_id, self.name = model_id, f"local-open-source:{model_id}"
        self.tokenizer = self.model = None

    def load(self):
        if self.model is not None: return 0.0
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        started = time.perf_counter()
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model = AutoModelForCausalLM.from_pretrained(self.model_id, dtype="auto", low_cpu_mem_usage=True)
        self.model.eval()
        self.model.generation_config.temperature = None
        self.model.generation_config.top_p = None
        self.model.generation_config.top_k = None
        return time.perf_counter() - started

    def _generate_once(self, system, user, schema, repair=""):
        self.load()
        import torch
        task = schema.get("task")
        messages = [{"role": "system", "content": system +
                     " Return one complete JSON object only. Include every contract field."},
                    {"role": "user", "content": user + "\nContract: " +
                     json.dumps(schema, ensure_ascii=False) + repair}]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.inference_mode():
            token_budget = {"plan": 48, "judge": 32}.get(task, 64)
            out = self.model.generate(**inputs, max_new_tokens=token_budget, do_sample=False,
                                      repetition_penalty=1.05, pad_token_id=self.tokenizer.eos_token_id)
        return extract_json(self.tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

    def _generate_text(self, user, instruction, max_new_tokens=48):
        import torch
        messages = [{"role": "system", "content": instruction + " Return one concise sentence only."},
                    {"role": "user", "content": user}]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.inference_mode():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      repetition_penalty=1.05,
                                      pad_token_id=self.tokenizer.eos_token_id)
        text = self.tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        text = text.splitlines()[0].strip().strip('"')
        if not text: raise ValueError("local model returned an empty coaching field")
        return text

    def _select_score(self, user, dimension):
        import torch
        messages = [{"role": "system", "content":
                     "You are a rubric scorer. Choose one integer from 1 (weak) to 5 (strong)."},
                    {"role": "user", "content": user +
                     f"\nDimension: {dimension}. Answer with exactly one digit. Score:"}]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.inference_mode(): logits = self.model(**inputs).logits[0, -1]
        candidates = {}
        for score in range(1, 6):
            token_ids = set()
            for form in (str(score), " " + str(score)):
                encoded = self.tokenizer.encode(form, add_special_tokens=False)
                if encoded: token_ids.add(encoded[0])
            candidates[score] = max(float(logits[token_id]) for token_id in token_ids)
        return float(max(candidates, key=candidates.get))

    def _generate_plan(self, user):
        """由本地模型在允许工具中选择，Python 只负责数字与工具名映射。"""
        import torch
        options = {1: "text_analysis", 2: "rubric_lookup", 3: "evidence_locator"}
        messages = [
            {"role": "system", "content": "Choose the most useful first tool for this Writing Coach task."},
            {"role": "user", "content": user +
             "\n1=text_analysis, 2=rubric_lookup, 3=evidence_locator. Answer one digit only:"},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.inference_mode(): logits = self.model(**inputs).logits[0, -1]
        strengths = {}
        for number in options:
            ids = self.tokenizer.encode(str(number), add_special_tokens=False)
            if ids: strengths[number] = float(logits[ids[0]])
        selected = options[max(strengths, key=strengths.get)]
        reason = self._generate_text(user, f"Explain briefly why {selected} is useful for this task.", 16)
        return {"goal": "model-selected grounded diagnosis",
                "steps": [{"tool": selected, "reason": reason}],
                "_structured_from_local_model_fields": True}

    def _generate_report(self, user):
        language_score = self._select_score(user, "language clarity, grammar, cohesion")
        argument_score = self._select_score(user, "claim, reasons, evidence, counterargument")
        coaching = self._generate_text(
            user,
            "In one short sentence, diagnose the most important language or argument issue and give one revision action without rewriting the essay.",
            24,
        )
        evidence_id = self._select_evidence_id(user)
        return {
            "summary": coaching,
            "scores": {
                "language": {"score": language_score, "reason": coaching, "evidence_ids": [evidence_id]},
                "argumentation": {"score": argument_score, "reason": coaching, "evidence_ids": [evidence_id]},
            },
            "priorities": [{"issue": coaching, "evidence_id": evidence_id,
                            "action": coaching, "example": "For example, ... This shows that ..."}],
            "revision_plan": [coaching],
            "highlights": [{"sentence_id": evidence_id, "label": "needs_evidence", "reason": coaching}],
            "_structured_from_local_model_fields": True,
        }

    def _select_evidence_id(self, user):
        """在实际句号候选中比较模型 logits，避免固定引用第 1 句。"""
        import torch
        try:
            payload = json.loads(user)
            rows = payload.get("facts", {}).get("evidence_locator", [])
            valid_ids = [int(row["sentence_id"]) for row in rows][:9]
        except Exception:
            valid_ids = []
        if not valid_ids:
            return 1
        messages = [
            {"role": "system", "content": "Choose the one sentence ID that most needs evidence or explanation."},
            {"role": "user", "content": user + f"\nValid IDs: {valid_ids}. Answer one digit only. ID:"},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt")
        with torch.inference_mode():
            logits = self.model(**inputs).logits[0, -1]
        strengths = {}
        for sentence_id in valid_ids:
            ids = self.tokenizer.encode(str(sentence_id), add_special_tokens=False)
            if ids:
                strengths[sentence_id] = float(logits[ids[0]])
        return max(strengths, key=strengths.get) if strengths else valid_ids[0]

    def _generate_judge(self, user):
        """Judge 的评分和理由由模型生成，Python 只将 3 分作为通过阈值。"""
        score = self._select_score(user, "grounding, actionability, teaching value, and no full-essay rewriting")
        audit = self._generate_text(
            user, "In one short sentence, judge feedback quality and name its most important remaining risk.", 20)
        return {"pass": score >= 3, "score": score, "reason": audit, "risks": [audit],
                "_structured_from_local_model_fields": True}

    def generate_json(self, system, user, schema):
        if schema.get("task") == "plan":
            self.load()
            return self._generate_plan(user)
        if schema.get("task") == "report":
            self.load()
            return self._generate_report(user)
        if schema.get("task") == "judge":
            self.load()
            return self._generate_judge(user)
        result = self._generate_once(system, user, schema)
        return result

In [13]:
class ClassroomBackend:
    name = "classroom-fallback-rules (NOT AI)"
    def generate_json(self, system, user, schema):
        task = schema["task"]
        essay = user.lower()
        stronger = "for example" in essay and "however" in essay and "therefore" in essay
        if task == "plan":
            return {"goal": "按量表和原文证据完成 Writing Coach 诊断", "steps": [
                {"tool": "text_analysis", "reason": "读取带句号的文本事实"},
                {"tool": "rubric_lookup", "reason": "读取当前评分量表"},
                {"tool": "evidence_locator", "reason": "把判断绑定到原文句子"},
            ]}
        if task == "report":
            base = 4.1 if stronger else 2.3
            return {
                "summary": "论证结构较完整。" if stronger else "立场明确，但理由、例证与反方回应不足。",
                "scores": {
                    "language": {"score": base + .2, "reason": "按清晰度、句式与衔接综合判断。", "evidence_ids": [1]},
                    "argumentation": {"score": base, "reason": "按主张、理由、例证与回应综合判断。", "evidence_ids": [1]},
                },
                "priorities": [{"issue": "补强证据链", "evidence_id": 1,
                    "action": "补充具体例子，并解释它如何支持主张。", "example": "For example, ... This shows ..."}],
                "revision_plan": ["补具体例子", "解释例子", "回应一个反方观点"],
                "highlights": [{"sentence_id": 1, "label": "strength" if stronger else "needs_evidence",
                                "reason": "主张清楚" if stronger else "主张后需要证据"}],
            }
        if task == "judge":
            return {"pass": True, "score": 4, "reason": "建议引用原文并给出可执行动作。", "risks": ["当前为课堂降级结果"]}
        raise ValueError(task)

In [14]:
class SafeBackend:
    def __init__(self, primary, fallback):
        self.primary, self.fallback, self.name = primary, fallback, primary.name
        self.degraded, self.last_error = False, None
    def generate_json(self, system, user, schema):
        if self.degraded:
            return self.fallback.generate_json(system, user, schema)
        try:
            result = self.primary.generate_json(system, user, schema)
            self.name = self.primary.name
            return result
        except Exception as exc:
            self.force_fallback(f"{type(exc).__name__}: {exc}")
            return self.fallback.generate_json(system, user, schema)

    def force_fallback(self, error: str) -> None:
        self.degraded, self.last_error = True, error
        self.name = self.fallback.name
        print("⚠️ 本地模型输出连续校验失败，切换课堂降级后端:", error)

    def reset(self):
        self.degraded, self.last_error = False, None
        self.name = self.primary.name


local_model = LocalQwenBackend(MODEL_ID)
backend = SafeBackend(local_model, ClassroomBackend()) if RUN_LOCAL_MODEL else ClassroomBackend()
if RUN_LOCAL_MODEL:
    print(f"模型加载耗时: {local_model.load():.2f}s；后续 block 复用同一对象")
else:
    print("快速课堂模式；把第一个代码单元格的 RUN_LOCAL_MODEL 改为 True 可启用真实本地 AI")
print("当前后端:", backend.name)

快速课堂模式；把第一个代码单元格的 RUN_LOCAL_MODEL 改为 True 可启用真实本地 AI
当前后端: classroom-fallback-rules (NOT AI)


In [ ]:
def split_sentences(text: str) -> list[str]:
    return [x.strip() for x in re.split(r"(?<=[.!?])\s+", text.strip()) if x.strip()]

def text_analysis(essay: str) -> dict[str, Any]:
    return {"word_count": len(re.findall(r"\b[A-Za-z']+\b", essay)),
            "sentences": [{"id": i, "text": s} for i, s in enumerate(split_sentences(essay), 1)]}

def rubric_lookup() -> dict[str, Any]:
    return json.loads((DATA_DIR / "rubric.jsonl").read_text(encoding="utf-8"))

def evidence_locator(essay: str) -> list[dict[str, Any]]:
    labels = {"claim": r"\bshould\b", "reason": r"\bbecause\b", "example": r"\bfor example\b",
              "counterargument": r"\b(some people|however)\b", "conclusion": r"\btherefore\b"}
    return [{"sentence_id": i, "text": s,
             "labels": [k for k, p in labels.items() if re.search(p, s, re.I)]}
            for i, s in enumerate(split_sentences(essay), 1)]

PLAN_SCHEMA = {"task": "plan", "goal": "string",
    "steps": [{"tool": "text_analysis|rubric_lookup|evidence_locator", "reason": "string"}]}
REPORT_SCHEMA = {"task": "report", "summary": "string", "scores": {
        "language": {"score": "1..5", "reason": "string", "evidence_ids": [1]},
        "argumentation": {"score": "1..5", "reason": "string", "evidence_ids": [1]}},
    "priorities": [{"issue": "string", "evidence_id": 1, "action": "string", "example": "fragment"}],
    "revision_plan": ["action"],
    "highlights": [{"sentence_id": 1, "label": "strength|needs_evidence|language|counterargument", "reason": "string"}]}

In [16]:
@dataclass
class RevisionRound:
    round_id: int
    prompt: str
    essay: str
    run_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    facts: dict[str, Any] = field(default_factory=dict)
    report: dict[str, Any] | None = None
    trace: list[dict[str, Any]] = field(default_factory=list)

class WritingCoachAgent:
    ALLOWED_TOOLS = {"text_analysis", "rubric_lookup", "evidence_locator"}
    def __init__(self, backend: JSONBackend, output_dir: Path):
        self.backend, self.output_dir = backend, output_dir
        output_dir.mkdir(parents=True, exist_ok=True)

    def _validate_plan(self, raw: dict[str, Any]) -> list[dict[str, str]]:
        plan, seen = [], set()
        steps = raw.get("steps", [])
        # 小模型常把单步计划写成 {tool, reason}；语义合法时先标准化。
        if not steps and raw.get("tool"):
            steps = [raw]
        for item in steps:
            tool = str(item.get("tool", "")).strip()
            if tool in self.ALLOWED_TOOLS and tool not in seen:
                plan.append({"tool": tool, "reason": str(item.get("reason", ""))})
                seen.add(tool)
        if not plan:
            raise ValueError("Plan 必须至少包含一个白名单工具；字段名必须是 steps")
        # Rubric 与 Evidence 是安全底线；Planner 决定其余工具，Executor 补齐强制 grounding。
        for required, reason in [
            ("rubric_lookup", "Executor guardrail：评分前必须读取量表"),
            ("evidence_locator", "Executor guardrail：反馈必须绑定原文证据"),
        ]:
            if required not in seen:
                plan.append({"tool": required, "reason": reason})
                seen.add(required)
        return plan

    def _validate_report(self, report: dict[str, Any], sentence_count: int) -> None:
        if set(report.get("scores", {})) != {"language", "argumentation"}:
            raise ValueError("scores 必须同时包含 language 和 argumentation")
        for dimension, item in report["scores"].items():
            if not 1 <= float(item["score"]) <= 5:
                raise ValueError(f"{dimension} 分数越界")
            if not item.get("reason") or not item.get("evidence_ids"):
                raise ValueError(f"{dimension} 缺少 reason 或 evidence_ids")
            if any(not 1 <= int(i) <= sentence_count for i in item["evidence_ids"]):
                raise ValueError(f"{dimension} 的 evidence_ids 越界")
        if not report.get("priorities") or not report.get("revision_plan"):
            raise ValueError("报告必须包含 priorities 和 revision_plan")
        for mark in report.get("highlights", []):
            if not 1 <= int(mark["sentence_id"]) <= sentence_count:
                raise ValueError("高亮句号越界")

    def _normalize_report(self, raw: dict[str, Any], sentence_count: int) -> dict[str, Any]:
        """保留 AI 主体内容，只修复小模型常见的字段别名或辅助字段遗漏。"""
        report = json.loads(json.dumps(raw, ensure_ascii=False))
        notes = []
        scores = report.get("scores")
        if not isinstance(scores, dict):
            raise ValueError("报告缺少 scores 对象")
        priority = (report.get("priorities") or [{}])[0]
        fallback_evidence = int(priority.get("evidence_id", 1))
        fallback_evidence = min(max(fallback_evidence, 1), sentence_count)
        for dimension in ("language", "argumentation"):
            item = scores.get(dimension)
            if not isinstance(item, dict) or "score" not in item:
                raise ValueError(f"scores 缺少 {dimension}.score")
            if not item.get("reason") and item.get("rationale"):
                item["reason"] = item["rationale"]
                notes.append(f"{dimension}: rationale → reason")
            if not item.get("reason") and report.get("summary"):
                item["reason"] = report["summary"]
                notes.append(f"{dimension}: 使用 AI summary 补齐 reason")
            if not item.get("evidence_ids") and item.get("evidence_sentence_ids"):
                item["evidence_ids"] = item["evidence_sentence_ids"]
                notes.append(f"{dimension}: evidence_sentence_ids → evidence_ids")
            if not item.get("evidence_ids"):
                item["evidence_ids"] = [fallback_evidence]
                notes.append(f"{dimension}: 使用 priority 句号补齐 evidence_ids")
        if not report.get("highlights") and report.get("priorities"):
            report["highlights"] = [{
                "sentence_id": fallback_evidence,
                "label": "needs_evidence",
                "reason": str(priority.get("issue", report.get("summary", "需要关注"))),
            }]
            notes.append("使用 AI priority 生成 highlight 映射")
        if notes:
            report["schema_repaired"] = True
            report["schema_repair_notes"] = notes
        self._validate_report(report, sentence_count)
        return report

    def _validated_call(self, run: RevisionRound, stage: str, system: str, user: str,
                        schema: dict[str, Any], validator):
        """模型输出可解析但不合约时，带具体错误重试一次；仍失败才显式降级。"""
        last_error = ""
        attempts = (1, 2) if stage == "planning" else (1,)
        for attempt in attempts:
            run.trace.append({"event": "llm_started", "stage": stage, "attempt": attempt,
                              "backend": self.backend.name})
            repair = f"\nPrevious output failed validation: {last_error}. Repair it." if last_error else ""
            result = self.backend.generate_json(system, user + repair, schema)
            try:
                validated = validator(result)
                run.trace.append({"event": "llm_succeeded", "stage": stage, "attempt": attempt,
                                  "backend": self.backend.name})
                return result, validated
            except (KeyError, TypeError, ValueError) as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                run.trace.append({"event": "schema_validation_failed", "stage": stage,
                                  "attempt": attempt, "error": last_error,
                                  "raw_output": result})
        if isinstance(self.backend, SafeBackend):
            self.backend.force_fallback(f"{stage}: {last_error}")
            result = self.backend.generate_json(system, user, schema)
            return result, validator(result)
        raise ValueError(f"{stage} 连续两次输出不合法: {last_error}")

    def diagnose(self, round_id: int, prompt: str, essay: str) -> RevisionRound:
        if isinstance(self.backend, SafeBackend): self.backend.reset()
        if not essay.strip(): raise ValueError("作文不能为空")
        run = RevisionRound(round_id, prompt, essay)
        plan_raw, plan = self._validated_call(
            run, "planning",
            "You are the Planner of a Writing Coach Agent. Select tools before seeing their results. Use exact tool names from the contract.",
            f"Writing prompt: {prompt}\nEssay excerpt: {essay[:600]}\n"
            f"Available tools: {sorted(self.ALLOWED_TOOLS)}",
            PLAN_SCHEMA, self._validate_plan)
        run.trace.append({"event": "plan_created", "goal": plan_raw.get("goal", ""), "plan": plan})

        for step in plan:
            tool = step["tool"]
            run.trace.append({"event": "tool_started", "tool": tool, "reason": step["reason"]})
            if tool == "text_analysis": run.facts[tool] = text_analysis(essay)
            elif tool == "rubric_lookup": run.facts[tool] = rubric_lookup()
            elif tool == "evidence_locator": run.facts[tool] = evidence_locator(essay)
            run.trace.append({"event": "tool_succeeded", "tool": tool})
        run.trace.append({"event": "tools_completed", "tools": list(run.facts)})
        payload = json.dumps({"prompt": prompt, "essay": essay, "facts": run.facts}, ensure_ascii=False)
        raw_report, report = self._validated_call(
            run, "coaching",
            "You are a Writing Coach. Use only supplied rubric and sentence evidence. Score both dimensions, cite valid sentence IDs, and give actionable priorities without rewriting the whole essay.",
            payload, REPORT_SCHEMA,
            lambda value: self._normalize_report(value, len(split_sentences(essay))))
        if report.get("schema_repaired"):
            run.trace.append({"event": "schema_repaired", "stage": "coaching",
                              "notes": report["schema_repair_notes"], "raw_output": raw_report})
        report["plan"] = plan
        report["model_backend"] = self.backend.name
        report["degraded"] = getattr(self.backend, "degraded", False) or "NOT AI" in self.backend.name
        run.report = report
        run.trace += [{"event": "agent_report_completed", "backend": self.backend.name},
                      {"event": "checkpoint_saved", "run_id": run.run_id}]
        (self.output_dir / f"{run.run_id}.jsonl").write_text(
            json.dumps(asdict(run), ensure_ascii=False, indent=2), encoding="utf-8")
        return run

In [17]:
coach: WritingCoachAgent = WritingCoachAgent(backend, OUTPUT_DIR / "revision_runs_inline")

partial_ai_report = {
    "summary": "The claim is clear but needs more explanation.",
    "scores": {"language": {"score": 3.0}, "argumentation": {"score": 2.5}},
    "priorities": [{"issue": "evidence is too general", "evidence_id": 1,
                    "action": "add one concrete example", "example": "For example, ..."}],
    "revision_plan": ["add a concrete example", "explain how it supports the claim"],
}
partial_repaired = coach._normalize_report(partial_ai_report, sentence_count=1)
assert partial_repaired["schema_repaired"] is True
assert partial_repaired["scores"]["language"]["reason"]
assert partial_repaired["scores"]["argumentation"]["evidence_ids"] == [1]
print("✅ AI 报告辅助字段缺失回归测试通过：保留模型主体内容并透明修复 Schema")

✅ AI 报告辅助字段缺失回归测试通过：保留模型主体内容并透明修复 Schema


In [18]:
prompt = "Should students receive cash rewards for good grades?"
draft_1 = "Students should get money for grades. Money is useful. Good grades are good. Parents can give money and students will happy. This is my opinion."
draft_2 = "Students should not depend on cash rewards for good grades because learning should build long-term motivation. For example, a student may stop studying when a payment disappears. Some people argue that money creates quick motivation; however, feedback and opportunities can recognize progress without turning learning into a transaction. Therefore, schools should reward improvement with meaningful recognition rather than cash."

round_1 = coach.diagnose(1, prompt, draft_1)
print(f"第 1 轮 | backend={round_1.report['model_backend']} | degraded={round_1.report['degraded']}")
print("Plan:", round_1.report["plan"])
print("Scores:", json.dumps(round_1.report["scores"], ensure_ascii=False))
print("Revision plan:", round_1.report["revision_plan"])

第 1 轮 | backend=classroom-fallback-rules (NOT AI) | degraded=True
Plan: [{'tool': 'text_analysis', 'reason': '读取带句号的文本事实'}, {'tool': 'rubric_lookup', 'reason': '读取当前评分量表'}, {'tool': 'evidence_locator', 'reason': '把判断绑定到原文句子'}]
Scores: {"language": {"score": 2.5, "reason": "按清晰度、句式与衔接综合判断。", "evidence_ids": [1]}, "argumentation": {"score": 2.3, "reason": "按主张、理由、例证与回应综合判断。", "evidence_ids": [1]}}
Revision plan: ['补具体例子', '解释例子', '回应一个反方观点']
